# Objetivo
Este notebook utiliza las predicciones generadas por el modelo de demanda para calcular la cantidad óptima de pedido por producto y tienda, equilibrando el costo de desabastecimiento (stockout) y el costo de sobreinventario (overstock).

In [39]:
pip install pulp

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
You should consider upgrading via the 'c:\Users\beltr\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [1]:
import sys
from pathlib import Path

# Raíz del proyecto
project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

In [18]:
import importlib
import src.optimization
importlib.reload(src.optimization)

<module 'src.optimization' from 'c:\\Users\\beltr\\Documents\\prueba_tostao\\src\\optimization.py'>

In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pulp import *
from src.optimization import *

In [5]:
forecast = load_forecast("../data/processed/next_week_forecast.csv")
inventory = load_inventory("../data/raw/inventario_actual.csv")
catalog = load_catalog("../data/raw/catalogo_productos.csv")

In [6]:
df = prepare_optimization_data(
    forecast,
    inventory,
    catalog
)

In [7]:
df.head(2)

,id_tienda,id_producto,demanda_semanal,stock_actual,costo_unitario,precio_venta,costo_almacenamiento_semanal
0,STORE_01,PROD_001,52.302450,1,800,2500,10
1,STORE_01,PROD_002,69.475568,10,1200,3500,15


In [8]:
print(df.shape)

(160, 7)


In [9]:
df.isnull().sum()

id_tienda                       0
id_producto                     0
demanda_semanal                 0
stock_actual                    0
costo_unitario                  0
precio_venta                    0
costo_almacenamiento_semanal    0
dtype: int64

In [14]:
df = calculate_inventory_targets(df)

In [15]:
results = optimize_inventory(df) 
results.head()

,id_tienda,id_producto,demanda_semanal,stock_actual,costo_unitario,precio_venta,costo_almacenamiento_semanal,stock_seguridad,demanda_objetivo,pedido_recomendado,stock_final_estimado,costo_almacenamiento_total
0,STORE_01,PROD_001,52.302450,1,800,2500,10,10.460490,62.762940,62.0,10.697550,620.0
1,STORE_01,PROD_002,69.475568,10,1200,3500,15,13.895114,83.370682,74.0,14.524432,1110.0
2,STORE_01,PROD_003,69.729050,33,1500,4500,15,13.945810,83.674860,51.0,14.270950,765.0
3,STORE_01,PROD_004,51.128935,11,1000,2800,20,10.225787,61.354722,51.0,10.871065,1020.0
4,STORE_01,PROD_005,75.802040,3,800,2500,20,15.160408,90.962448,88.0,15.197960,1760.0


In [20]:
save_optimization_results(
    results,
    "../data/processed/optimization_results.csv"
)